In [1]:
import ccxt
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import datetime
bybit = ccxt.bybit()
from freqtrade_client import FtRestClient
from datetime import datetime, date, timedelta, timezone, time
import pandas as pd
import os
server_url = 'http://127.0.0.1:8080'
username = ''
password = ""
client = FtRestClient(server_url, username, password)

In [21]:
trades = pd.DataFrame(client.trades().get('trades'))

starting_balance = client.daily(1).get('data')[0].get('starting_balance')
today_loss = trades[
    (trades.close_profit_abs < 0) & 
    (trades.close_date > date.today().strftime('%Y-%m-%d'))
].close_profit_abs.sum().item() / starting_balance

week_day = date.weekday(date.today())
open_date = (date.today() - timedelta(days=week_day))
starting_balance = client.weekly(1).get('data')[0].get('starting_balance')
this_week_loss = trades[
    (trades.close_profit_abs < 0) & 
    (trades.close_date > open_date.strftime('%Y-%m-%d'))
].close_profit_abs.sum().item() / starting_balance

In [24]:
today_loss

-0.010614419988958724

In [20]:
today_loss

-3.15281739

In [11]:
week_day = date.weekday(date.today())
open_date = (date.today() - timedelta(days=week_day))
open_date

datetime.date(2024, 11, 18)

In [ ]:
all_trades = client.trades().get('trades') + client.status()
dataframe = pd.DataFrame(all_trades)
dataframe.is_open.any()

np.True_

In [27]:
stoploss = -0.01
max_stake = 300
min_stake = 5
risk = 0.01
max(min(abs(stoploss) / 2 * max_stake / risk, max_stake), min_stake)

150.0

In [28]:
1.2/1.5

0.7999999999999999

In [36]:
270/180

1.5

In [ ]:
risk = np.arange(0, 1, 0.001)
risk * 300 / 1.5

array([  0. ,   0.2,   0.4,   0.6,   0.8,   1. ,   1.2,   1.4,   1.6,
         1.8,   2. ,   2.2,   2.4,   2.6,   2.8,   3. ,   3.2,   3.4,
         3.6,   3.8,   4. ,   4.2,   4.4,   4.6,   4.8,   5. ,   5.2,
         5.4,   5.6,   5.8,   6. ,   6.2,   6.4,   6.6,   6.8,   7. ,
         7.2,   7.4,   7.6,   7.8,   8. ,   8.2,   8.4,   8.6,   8.8,
         9. ,   9.2,   9.4,   9.6,   9.8,  10. ,  10.2,  10.4,  10.6,
        10.8,  11. ,  11.2,  11.4,  11.6,  11.8,  12. ,  12.2,  12.4,
        12.6,  12.8,  13. ,  13.2,  13.4,  13.6,  13.8,  14. ,  14.2,
        14.4,  14.6,  14.8,  15. ,  15.2,  15.4,  15.6,  15.8,  16. ,
        16.2,  16.4,  16.6,  16.8,  17. ,  17.2,  17.4,  17.6,  17.8,
        18. ,  18.2,  18.4,  18.6,  18.8,  19. ,  19.2,  19.4,  19.6,
        19.8,  20. ,  20.2,  20.4,  20.6,  20.8,  21. ,  21.2,  21.4,
        21.6,  21.8,  22. ,  22.2,  22.4,  22.6,  22.8,  23. ,  23.2,
        23.4,  23.6,  23.8,  24. ,  24.2,  24.4,  24.6,  24.8,  25. ,
        25.2,  25.4,

In [47]:
.55*270

148.5

In [3]:
def return_order_book(symbol='BTC/USDT', n=200):
    bybit_ob = bybit.fetchOrderBook(symbol, n)
    binance_ob = bybit.fetchOrderBook(symbol, n)
    kucoin_ob = bybit.fetchOrderBook(symbol, n)
    bid_values = {
        'price': np.hstack((np.array(bybit_ob['bids'])[:,0], np.array(binance_ob['bids'])[:,0], np.array(kucoin_ob['bids'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['bids'])[:,1], np.array(binance_ob['bids'])[:,1], np.array(kucoin_ob['bids'])[:,1])),
        'side':'bid'
    }
    ask_values = {
        'price': np.hstack((np.array(bybit_ob['asks'])[:,0], np.array(binance_ob['asks'])[:,0], np.array(kucoin_ob['asks'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['asks'])[:,1], np.array(binance_ob['asks'])[:,1], np.array(kucoin_ob['asks'])[:,1])),
        'side':'ask'
    }
    bid_dataframe = pd.DataFrame(bid_values)
    ask_dataframe = pd.DataFrame(ask_values)
    dataframe = pd.concat((bid_dataframe,ask_dataframe))
    dataframe = dataframe.groupby(['price','side']).sum().reset_index()
    # dataframe.groupby('side').sum()
    dataframe['now'] = datetime.datetime.now()
    return dataframe

In [3]:
def plot(dataframe, trades=[]):

    fig = go.Figure(data=[go.Candlestick(x=dataframe.date.values,
                    open=dataframe['open'],
                    high=dataframe['high'],
                    low=dataframe['low'],
                    close=dataframe['close'],
                    increasing_line_color= 'green', 
                    decreasing_line_color= 'red')])

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.upper_band.values,
        mode="lines", 
        marker=dict(size=7, color="green")
    )

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.y.values,
        mode="lines", 
        marker=dict(size=7, color="blue")
    )

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.lower_band.values,
        mode="lines", 
        marker=dict(size=7, color="red")
    )

    fig.add_scatter(
        x= dataframe[dataframe.extrema == 1].date.values, 
        y= dataframe[dataframe.extrema == 1].high.values,
        mode="markers", 
        marker=dict(size=7, color="purple")
    )

    fig.add_scatter(
        x= dataframe[dataframe.extrema == -1].date.values, 
        y= dataframe[dataframe.extrema == -1].low.values,
        mode="markers", 
        marker=dict(size=7, color="yellow")
    )

    if not trades.empty:
        fig.add_scatter(
            x= trades.open_date.values, 
            y= trades.open_rate.values,
            mode="markers", 
            marker=dict(size=15, color="green")
        )

        fig.add_scatter(
            x= trades.close_date.values, 
            y= trades.close_rate.values,
            mode="markers", 
            marker=dict(size=15, color="red")
        )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    fig.update_layout(autosize=True, height=500,xaxis_rangeslider_visible=False)
    fig.show()

In [4]:
def calculate_extrema(dataframe, kernel=6):
    dataframe["extrema"] = 0
    min_peaks = argrelextrema(dataframe["low"].values, np.less_equal, order=kernel)
    max_peaks = argrelextrema(dataframe["high"].values, np.greater_equal, order=kernel)
    for mp in min_peaks[0]:
        dataframe.at[mp, "extrema"] = -1
    for mp in max_peaks[0]:
        dataframe.at[mp, "extrema"] = 1
    dataframe['last_min_peak'] = dataframe.at[min_peaks[0][-1], "low"]
    dataframe['last_max_peak'] = dataframe.at[max_peaks[0][-1], "high"]
    dataframe['h_dist'] = np.where(dataframe.extrema == 1, (dataframe.high - dataframe.upper_band), 0)
    dataframe['l_dist'] = np.where(dataframe.extrema == -1, (dataframe.lower_band - dataframe.low), 0)
    dataframe['h_ratio'] = dataframe['h_dist'] / dataframe['band_dist']
    dataframe['l_ratio'] = dataframe['l_dist'] / dataframe['band_dist']
    dataframe['l_h_ratio'] = dataframe.at[max_peaks[0][-1], "h_ratio"]
    dataframe['l_l_ratio'] = dataframe.at[min_peaks[0][-1], "l_ratio"]
    dataframe['last_max'] = dataframe.at[max_peaks[0][-1], "close"]
    dataframe['last_min'] = dataframe.at[min_peaks[0][-1], "close"]
    return dataframe

In [5]:
def caculate_regression(dataframe, kernel=1440):
    dataframe_ = dataframe.copy()[-kernel:]
    x = dataframe_.index.values.reshape(-1, 1)
    y = dataframe_.close.values
    model = LinearRegression()
    model.fit(x, y)
    x = dataframe.index.values.reshape(-1, 1)
    dataframe['y'] = model.predict(x)
    dataframe['coef'] = float(model.coef_[0])
    dataframe['upper_band'] = dataframe['y'] + dataframe.high.std()
    dataframe['lower_band'] = dataframe['y'] - dataframe.low.std()
    dataframe['band_dist'] = dataframe['upper_band'] - dataframe['lower_band']
    return dataframe

In [ ]:
!docker-compose run --rm TradeStrategy download-data -c user_data/config.json --timeframe 1m

In [13]:
def return_dataframe_from_csv(pair, columns=[]):
    dataframe = pd.read_csv(f'df_{pair}.csv')
    dataframe['date'] = pd.to_datetime(dataframe['date'])
    if columns:
        dataframe = dataframe[columns]
    return dataframe

In [30]:
files = [file for file in os.listdir(".") if file.endswith('.csv')]
tickers = [file[3:-4] for file in files]

In [59]:
tickers

['SOL',
 'OP',
 'WIF',
 'BTC',
 'AVAX',
 'LINK',
 'GOAT',
 'ETH',
 'LTC',
 'XRP',
 'ADA',
 'PNUT',
 '1000PEPE',
 'DOT',
 'HBAR',
 'FTM',
 'SHIB1000',
 'SUI',
 '1000BONK',
 'DOGE',
 'XLM']

In [36]:
def plot_ticker(ticker):
# dataframe = pd.read_feather("../data/bybit/futures/BTC_USDT_USDT-1m-futures.feather")
    columns=['date','open','high','low','close','volume']
    dataframe = return_dataframe_from_csv(ticker)
    # open_time = '2024-11-19 12:42'
    # close_time = '2024-11-20 12:42'
    now = datetime.datetime.now()
    close_time = now.strftime("%Y-%m-%d %H:%M:%S")
    open_time = (now - datetime.timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
    # dataframe = dataframe[(dataframe.date > open_time) & (dataframe.date <= close_time)].reset_index()
    # dataframe = caculate_regression(dataframe, kernel=1440)
    # dataframe = calculate_extrema(dataframe, kernel=6)
    trades = pd.DataFrame(client.trades().get('trades') + client.status())
    trades = trades[(trades.open_date > open_time) & (trades.close_date <= close_time)].reset_index()
    last_candle = dataframe.iloc[-1].squeeze()
    print(float((last_candle['upper_band'] - last_candle['y']) / (last_candle['y'] - last_candle['lower_band'])))
    print("Coef:", dataframe.iloc[-1].squeeze()['coef'])
    print(ticker)
    plot(dataframe,trades=pd.DataFrame())

In [92]:
plot_ticker(tickers[7])

0.9954628152324212
Coef: 0.070988779070431
ETH


In [83]:
ETH_dataframe = return_dataframe_from_csv("ETH")

In [74]:
ETH_dataframe.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'y', 'coef',
       'upper_band', 'lower_band', 'band_dist', 'extrema', 'last_min_peak',
       'last_max_peak', 'h_dist', 'l_dist', 'h_ratio', 'l_ratio', 'l_h_ratio',
       'l_l_ratio', 'last_max', 'last_min', 'second_last_max',
       'second_last_min', 'enter_tag', 'enter_long', 'enter_short'],
      dtype='object')

In [84]:
ETH_dataframe[ETH_dataframe.extrema == 1].close.values

array([3428.64, 3389.7 , 3396.12, 3384.9 , 3370.37, 3374.05, 3374.99,
       3375.  , 3373.32, 3381.22, 3382.83, 3386.6 , 3384.31, 3378.29,
       3382.33, 3383.23, 3379.71, 3378.61, 3379.51, 3387.  , 3371.79,
       3370.6 , 3339.41, 3333.4 , 3348.28, 3345.51, 3347.61, 3344.52,
       3359.01, 3352.33, 3302.43, 3309.64, 3305.11, 3304.62, 3298.16,
       3309.56, 3325.19, 3301.58, 3306.59, 3301.09, 3316.95, 3321.36,
       3324.86, 3321.67, 3307.4 , 3301.92, 3308.15, 3292.71, 3292.52,
       3299.01, 3288.39, 3296.45, 3287.92, 3296.05, 3295.41, 3289.76,
       3291.39, 3316.  , 3311.61, 3314.82, 3321.63, 3315.71, 3316.7 ,
       3317.31, 3331.82, 3330.9 , 3330.01, 3367.56, 3360.65, 3357.33,
       3339.07, 3347.3 , 3349.99, 3345.41, 3335.51, 3332.26, 3332.4 ,
       3330.73, 3343.54, 3340.3 , 3346.99, 3353.2 , 3359.22, 3343.98,
       3363.45, 3355.4 , 3354.06, 3355.37, 3352.39, 3351.56, 3353.86,
       3343.01, 3347.31, 3400.48, 3387.11, 3372.15, 3369.97, 3369.54,
       3358.18, 3364

In [86]:
ETH_dataframe.y.values

array([3243.83349762, 3243.90293667, 3243.97237571, ..., 3382.50327243,
       3382.57271148, 3382.64215052])

In [93]:
trades

,index,trade_id,pair,base_currency,quote_currency,is_open,exchange,amount,amount_requested,stake_amount,...,is_short,trading_mode,funding_fees,amount_precision,price_precision,precision_mode,precision_mode_price,contract_size,has_open_orders,orders
0,0,1,BTC/USDT:USDT,BTC,USDT,False,bybit,0.002,0.002985,196.4352,...,False,futures,0.0,0.001,0.1,4,4,1.0,False,"[{'amount': 0.002, 'safe_price': 98217.6, 'ft_..."
1,1,2,BTC/USDT:USDT,BTC,USDT,False,bybit,0.002,0.002984,196.2172,...,False,futures,0.0,0.001,0.1,4,4,1.0,False,"[{'amount': 0.002, 'safe_price': 98108.6, 'ft_..."


In [ ]:
trades = pd.read_csv(f'trades.csv')
trade = trades.iloc[1].squeeze()

In [17]:
dataframe = pd.read_csv(f'XLM_12_df.csv')
dataframe['date'] = pd.to_datetime(dataframe['date'])

In [34]:
((dataframe.date >= trade.open_date) & (dataframe.close < 0.54523)).mean()

np.float64(0.05675675675675676)